# Exploring MLflow runs, experiments, and model registry interactively

I wanted to get hands-on with the three core MLflow concepts — **runs**, **experiments**, and the **Model Registry** — and see how they connect in practice. This notebook walks through creating experiments, logging runs with different parameters, searching and comparing them, then registering a model and promoting it through stages.

last_verified: 2026-07-09 · MLflow n/a

## Setup

I'll use a local SQLite tracking URI so nothing leaks out. The dataset is the wine quality dataset — small enough to iterate quickly.

In [ ]:
import mlflow
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

mlflow.set_tracking_uri("sqlite:///mlflow_explore.db")

In [ ]:
# Load data and split once — reuse across runs
data = load_wine()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

## Create an experiment

MLflow groups runs into experiments. If you don't create one, runs go into the default "0" experiment. I'll make a named one so it's easier to find later.

In [ ]:
experiment_name = "wine-classifier-comparison"
exp = mlflow.set_experiment(experiment_name)
print(f"Experiment ID: {exp.experiment_id}")

## Run 1: RandomForest with default params

I'll log params, metrics, and the model artifact manually so I can see exactly what goes into a run.

In [ ]:
with mlflow.start_run(run_name="rf-default") as run:
    run_id = run.info.run_id
    
    # Log parameters
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", None)
    
    # Train
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # Log metrics
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    mlflow.log_metric("precision_macro", precision_score(y_test, y_pred, average="macro"))
    mlflow.log_metric("recall_macro", recall_score(y_test, y_pred, average="macro"))
    mlflow.log_metric("f1_macro", f1_score(y_test, y_pred, average="macro"))
    
    # Log the model itself
    mlflow.sklearn.log_model(model, "model")
    
    print(f"Run {run.info.run_id} complete")

## Run 2: GradientBoosting with tuned params

Different model family, different params — this will make comparison more interesting.

In [ ]:
with mlflow.start_run(run_name="gbm-tuned") as run:
    mlflow.log_param("model_type", "GradientBoosting")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("max_depth", 3)
    
    model = GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    mlflow.log_metric("precision_macro", precision_score(y_test, y_pred, average="macro"))
    mlflow.log_metric("recall_macro", recall_score(y_test, y_pred, average="macro"))
    mlflow.log_metric("f1_macro", f1_score(y_test, y_pred, average="macro"))
    
    mlflow.sklearn.log_model(model, "model")
    print(f"Run {run.info.run_id} complete")

## Run 3: RandomForest with more trees

I want to see if more estimators helps the RF.

In [ ]:
with mlflow.start_run(run_name="rf-300-trees") as run:
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("max_depth", 10)
    
    model = RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    mlflow.log_metric("precision_macro", precision_score(y_test, y_pred, average="macro"))
    mlflow.log_metric("recall_macro", recall_score(y_test, y_pred, average="macro"))
    mlflow.log_metric("f1_macro", f1_score(y_test, y_pred, average="macro"))
    
    mlflow.sklearn.log_model(model, "model")
    print(f"Run {run.info.run_id} complete")

## Explore runs programmatically

Now I'll use the MLflow API to find and compare runs in the experiment. This is the part I was most curious about — how do you query runs after they're logged?

In [ ]:
# Search all runs in the experiment
runs = mlflow.search_runs(
    experiment_ids=[exp.experiment_id],
    order_by=["metrics.accuracy DESC"]
)

display_cols = [
    "run_id", "run_name",
    "params.model_type", "params.n_estimators",
    "metrics.accuracy", "metrics.f1_macro"
]
runs[display_cols]

Nice — the GBM run is on top. I can see all three runs sorted by accuracy.

Let me grab the best run's ID so I can register its model.

In [ ]:
best_run = runs.iloc[0]
best_run_id = best_run["run_id"]
best_accuracy = best_run["metrics.accuracy"]
print(f"Best run: {best_run['run_name']} (ID: {best_run_id}) — accuracy: {best_accuracy:.4f}")

## Register the best model

The Model Registry stores models with versions and stage assignments. I'll register the GBM model and give it a descriptive name.

In [ ]:
model_uri = f"runs:/{best_run_id}/model"
registered_name = "wine-classifier-gbm"

result = mlflow.register_model(model_uri, registered_name)
print(f"Registered model: {result.name}, version: {result.version}")

## Transition the model to Staging

Model Registry stages let you track promotion: None → Staging → Production → Archived. I'll move version 1 to Staging.

In [ ]:
client = mlflow.MlflowClient()

client.transition_model_version_stage(
    name=registered_name,
    version=1,
    stage="Staging"
)
print(f"Model {registered_name} version 1 moved to Staging")

## Add a description and load the model

I'll annotate the model version with a note about its performance, then load it to verify it works.

In [ ]:
client.update_model_version(
    name=registered_name,
    version=1,
    description=f"GradientBoosting with n_estimators=200, lr=0.05. Test accuracy: {best_accuracy:.4f}"
)
print("Description added")

In [ ]:
# Load the model from the registry and run a quick prediction
import mlflow.pyfunc

model_staging = mlflow.pyfunc.load_model(
    model_uri=f"models:/{registered_name}/Staging"
)
sample_pred = model_staging.predict(X_test[:3])
print(f"Sample predictions: {sample_pred}")

## What I'd try next

The full cycle from experiment → run → registry is clearer now. I'd like to try:
- Wrapping this into a proper `mlflow.autolog()` workflow to compare automated vs manual capture
- Setting up the MLflow UI (`mlflow ui`) and browsing these runs visually
- Adding a CI step that automatically promotes models based on a metric threshold

## Got stuck on

- The `search_runs` DataFrame columns have `params.` and `metrics.` prefixes — took me a minute to realize that's how you reference them
- `register_model` needs a `runs:/<run_id>/model` URI, not just the run ID
- Stage names are case-sensitive (`"Staging"` not `"staging"`)